# Leksjon 18: Sikring av AI-agenter med kryptografiske kvitteringer

## Praktisk notatbok

Denne notatboken går gjennom fire øvelser:

1. **Signer din første kvittering** for et agentverktøy-kall og verifiser den.
2. **Manipuler kvitteringen** og se verifiseringen mislykkes.
3. **Bygg en kjede av tre kvitteringer** og bekreft kjedens integritet.
4. **Pakk inn et Microsoft Agent Framework-verktøykall** slik at hver handling sender ut en kvittering.

Alle kryptografiske primitivene importeres fra godt vedlikeholdte biblioteker (`pynacl` for Ed25519, `jcs` for RFC 8785 kanonisk JSON, `hashlib` fra Python standardbibliotek for SHA-256). Kvitteringslogikken i seg selv er vanlig Python som du kan lese og endre.

Kjør cellene i rekkefølge. Hver seksjon er kort og selvstendig.


## Oppsett

Installer de to avhengighetene. Begge har åpne lisenser (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Hjelpeverktøy

Disse to hjelpefunksjonene håndterer base64url-koding (uten utfylling) og SHA-256-hashing av vilkårlige objekter. De holder resten av notatblokken fokusert på kvitteringslogikken selv.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Seksjon 1: Signer din første kvittering

Forestill deg at vår agent for **Contoso Travel** nettopp har søkt etter flyreiser fra Sydney til Los Angeles for en kunde. Vi ønsker å registrere gatewayens uttalelse om dette verktøyoppringet slik at en fremtidig revisor kan verifisere det under en separat festet utstedernøkkel.

### Trinn 1.1: Generer en signeringsnøkkel

I produksjon ville agentens signeringsnøkkel ligge i en hardware security module (HSM), Azure Key Vault eller et tilsvarende beskyttet lager. For denne leksjonen genererer vi en ny nøkkel i minnet.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Trinn 1.2: Bygg kvitteringsdataene

Dataene inneholder alt vi ønsker at kvitteringen skal bekrefte: hvem som handlet, hvilket verktøy, med hvilke argumenter, hva som kom tilbake, under hvilken policy, og når. Vi hasher argumentene og resultatet i stedet for å inkludere dem direkte, slik at kvitteringen ikke lekker sensitivt innhold.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Steg 1.3: Signer og sett sammen kvitteringen

Tre trinn:

1. Kanoniser nyttelasten ved å bruke JCS slik at to implementasjoner som produserer samme logiske kvittering produserer byte-identiske bytes.
2. Signer de kanoniske bytene direkte med Ed25519 private nøkkel. PureEdDSA hasher meldingen internt, så en ekstra forhåndshashing ville endret protokollen.

Signaturen festes deretter til den originale nyttelasten for å produsere den endelige kvitteringen.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Trinn 1.4: Verifiser kvitteringen

Verifisering reverserer prosessen. Vi fjerner signaturen, beregner de kanoniske bytene på nytt, og sjekker signaturen mot utstederens offentlige nøkkel.

Verifisering skjer offline: ingen tjeneste å kontakte og ingen nettverkstilgang. Den trenger fortsatt én ting i tillegg til kvitteringen: utstederens offentlige nøkkel, hentet gjennom en kanal som revisor allerede stoler på, som et JWK-sett utstederen publiserer eller en verdi som er forhåndslagret. Feltet `public_key` inne i kvitteringen kan ikke spille den rollen, fordi den som skriver kvitteringen også skriver det feltet. Seksjon 2 viser forfalskningen dette forhindrer.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Du bør se `Receipt is valid: True`, `Pre-hashed receipt valid: False` og `Malformed receipts rejected: True`. Det positive tilfellet beviser at den direkte JCS-signeringsveien fungerer; den negative kontrollen gjør signaturskala-regelen kjørbar i stedet for å la den være som tekst; den siste linjen viser ugyldig input som returnerer `False` i stedet for å kaste en unntak.


## Seksjon 2: Manipuler kvitteringen

Hele poenget med kvitteringer er at de skal være manipulering-sikre. La oss bevise det.

Vi vil endre nøyaktig ett tegn i kvitteringen og se at verifiseringen feiler.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Hva skjedde nettopp?

Når vi endret `policy_id`, endret de kanoniske byteverdiene seg. Signaturen (som var over de opprinnelige kanoniske byteverdiene) stemmer ikke lenger. Verifiseringen returnerer korrekt `False`.

Det er ikke mulig å endre et signert felt, altså noe utenfor `signature`-objektet, og fortsatt få kvitteringen til å verifisere mot utstederens offentlige nøkkel, med mindre angriperen har utstederens private nøkkel. `signature`-objektet selv er ikke signert: `sig` er signaturen, og `verify_receipt` ignorerer `public_key`, så endring av det feltet alene endrer ingenting. Så lenge den private nøkkelen er i en nøkkellager og verifikatorer bruker utstederens uavhengig publiserte nøkkel, er det umulig å skjule en endring i et signert felt. Den betingelsen er viktig: neste celle viser en modifisert kvittering som verifiserer perfekt mot en annen nøkkel.

Prøv det selv: endre `tool_name`, eller `agent_id`, eller `timestamp` i cellen over og kjør på nytt. Hver endring i et signert felt gir en ugyldig kvittering.


### Endringen som signaturen ikke fanger opp

Angriperen ovenfor skrev ikke på nytt. En mer forsiktig en ville: endret kvitteringen,
signert den med en nøkkel de genererte selv, og lagt den nøkkelen inn i
`public_key` feltet. Resultatet er en kvittering hvis signatur er helt ekte
for innholdet.

Kjør dette og se hvilken av de to kontrollene som oppdager det.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Du bør se `True` deretter `False`. De tre `assert` linjene gjør dette til en
sjekk: cellen kaster en feil hvis forfalskningen noen gang verifiseres mot
utstederens nøkkel.

Forfalskningen er internt konsistent. Signaturen samsvarer virkelig med
innholdet og nøkkelen den bærer, så en verifikator som leser nøkkelen fra
kvitteringen rapporterer den som gyldig. Ingenting ved kvitteringen ser galt ut, fordi
ingenting ved den er inkonsistent.

Det eneste som avslører den er å sjekke mot en nøkkel angriperen ikke hadde
noen mulighet til å påvirke. Dette er grunnen til at `verify_receipt` tar nøkkelen som et argument: en
funksjon med signaturen `verify_receipt(receipt)` kan ikke skrives sikkert,
fordi den eneste nøkkelen den kunne nå er den angriperen leverte.

Den samme kvitteringen lagres som `sample_receipts/04_resigned_receipt.json` hvis du
ønsker å undersøke den utenfor notatblokken.


## Seksjon 3: Kjed kvitteringer sammen

En enkelt kvittering beskytter én handling. De fleste agenter utfører mange handlinger. For å gjøre hele sekvensen manipulasjonssikker, lenker vi hver kvittering til den forrige ved å inkludere den forrige kvitteringens hash i den nye kvitteringens payload.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Hvis noen fjerner eller omorganiserer en kvittering i den presenterte sekvensen, brytes neste lenke. Å oppdage forkortelse eller signatorkontrollert omskriving av suffikset krever også et festet gjeldende hode, ekstern sjekkpunkt eller autentisert populasjonsforpliktelse.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Bryt nå kjeden ved å tukle med mottaket i midten og verifiser på nytt. Det tuklede mottaket feiler signatursjekken, OG det neste mottaket feiler sjekken av kjedelenken (fordi dets `previous_receipt_hash` ikke lenger samsvarer med hash-en til det endrede midtmottaket).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Kvitto 0 verifiserer fortsatt (det ble ikke endret og har ingen forgjenger å avhenge av). Kvitto 1 feiler signatursjekken fordi vi endret `tool_args_hash`. Kvitto 2 feiler kjedelink-sjekken fordi dens `previous_receipt_hash` ble beregnet mot det originale (nå endrede) kvitto 1.

Selv om en angriper signerer det endrede kvitto 1 på nytt, vil den nye signaturen feile mot utstederens nøkkel med mindre de har utstederens private nøkkel (en ny signatur med deres egen nøkkel feiler, som vist i seksjon 2), og kjedelink-mangelen i kvitto 2 vil fortsatt avsløre manipulasjonen. For å skjule endringen må angriperen signere på nytt hvert kvitto fra endringspunktet og fremover, noe som krever besittelse av utstederens private nøkkel.


## Seksjon 4: Pakk inn et agentverktøykall med kvitteringssignering

I en ekte utrulling ønsker du ikke at hver agentforfatter skal huske å kalle `make_receipt`. Du vil at kvitteringssignering skal være automatisk for hvert verktøykall.

Her er det enkleste mønsteret: en wrapper-klasse som tar hvilken som helst kallbar verktøyfunksjon og returnerer en versjon som utsteder kvitteringer. Dette tilpasses til enhver agentramme, inkludert Microsoft Agent Framework (`agent_framework.foundry`).

Hvis du ikke har et Microsoft Foundry-prosjekt satt opp, demonstrerer den lokale mocken nedenfor fortsatt mønsteret.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integrering med Microsoft Agent-rammeverket

Wrappen `ReceiptedTool` ovenfor er rammeverksuavhengig. For å bruke den inne i en agent bygget med Microsoft Agent Framework, registrer den innpakkede funksjonen som et verktøy. Et skisseeksempel (du vil erstatte mocken med en ekte Microsoft Foundry-verktøyregistrering):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Agent-rammeverket trenger ikke vite noe om kvitteringer. Kvitteringssignering pakkes rundt verktøyet, ikke festet inn i rammeverket. Slik legger du til proveniens til eksisterende agentkode uten å omskrive agenten.


## Oppsummering og utfordring

Du har:

- Generert et Ed25519-nøkkelpar.
- Bygget og signert en kvittering for et agentverktøy-kall.
- Verifisert kvitteringen offline ved kun å bruke utstederens offentlige nøkkel, oppnådd uavhengig av kvitteringen.
- Tuklet med en kvittering og observert verifikasjonsfeil.
- Bygget en hash-kjedet sekvens av tre kvitteringer.
- Tuklet med midten av kjeden og observert både signaturfeil og feil i kjedelenken.
- Pakket inn en verktøyfunksjon med automatisk kvitteringssignering.

**Utfordring.** Utvid kvitteringsskjemaet med et `request_id`-felt (en UUID for distribuert sporing). Oppdater `make_receipt` for å inkludere dette, og bekreft at kvitteringene fortsatt verifiseres fra start til slutt. Endre så feltet etter signering og bekreft at verifisering feiler. Dette tvinger deg til å internalisere hvordan hver byte av den kanoniske kodingen bidrar til signaturen.

**Viktig grense.** Under en festet nøkkel- og kjedekontekst beviser kvitteringer signaturtilskrivelse, innholdsintegritet og en lenke til den presenterte forgjengeren. De beviser IKKE eksekvering, kilde-sannhet, pålitelig tid, fullstendighet, policyvurdering eller korrekthet. Kvitteringer er et fundament. Styring er systemet du bygger oppå.

Les leksjonens README igjen med dette skillet i tankene. Den vanligste feilen team gjør med kvitteringer er å anta at "vi har kvitteringer" betyr "vi er styrt." Det gjør det ikke. Kvitteringer gjør agentadferd revisjonssikker. De gjør den ikke korrekt.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
